# 03 - Veri Ön İşleme ve Ortak Pipeline

**Sorumlu:** Kişi 3 (ad soyad yazılacak)

**Ödevdeki karşılığı:** 2. Veri Ön İşleme

In [22]:
# KURULUM: bu hucreyi her oturumun basinda bir kez calistir.
# Repoyu Colab'e indirir; boylece src/ortak.py kullanilabilir.
#import os
#REPO_URL = "https://github.com/zeyneppbaran/ml-proje.git"  # TODO: kendi repo adresiniz
#if not os.path.exists("/content/ml-proje"):
 #   !git clone {REPO_URL} /content/ml-proje
##   !git -C /content/ml-proje pull
#%cd /content/ml-proje

In [23]:
import os
from pathlib import Path

print("Çalışma klasörü:", Path.cwd())
print("Klasör içeriği:", os.listdir())

Çalışma klasörü: c:\content\ml-proje
Klasör içeriği: ['.git', '.gitignore', 'figures', 'notebooks', 'rapor', 'README.md', 'requirements.txt', 'results', 'src']


In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Kütüphaneler başarıyla yüklendi!")

Kütüphaneler başarıyla yüklendi!


In [28]:
import pandas as pd
from tkinter import Tk, filedialog

# CSV dosyasını bilgisayardan seç
root = Tk()
root.withdraw()
root.attributes('-topmost', True)

dosya_yolu = filedialog.askopenfilename(
    title="Bank Marketing CSV dosyasını seç",
    filetypes=[("CSV Dosyaları", "*.csv")]
)

root.destroy()

if not dosya_yolu:
    raise FileNotFoundError("Herhangi bir dosya seçilmedi!")

# Veri setini oku
df = pd.read_csv(dosya_yolu, sep=';')

# 1. Veri boyutu
print("VERİ BOYUTU:")
print(df.shape)

# 2. Sütun bilgileri
print("\nSÜTUN BİLGİLERİ:")
df.info()

# 3. Boş değerler
print("\nBOŞ DEĞERLER:")
print(df.isnull().sum())

# 4. Unknown değerleri
print("\nUNKNOWN DEĞERLER:")
print((df == "unknown").sum())

# 5. Tekrarlanan satırlar
print("\nTEKRARLANAN SATIR SAYISI:")
print(df.duplicated().sum())

VERİ BOYUTU:
(41188, 21)

SÜTUN BİLGİLERİ:
<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17 

## D1. Eksik ve tutarsız veriler

In [29]:
print("Eksik değerlerin toplamı:")
print(df.isnull().sum().sum())

print("\nUnknown değerlerin sütunlara göre dağılımı:")
unknown_sayilari = (df == "unknown").sum()
print(unknown_sayilari[unknown_sayilari > 0])

Eksik değerlerin toplamı:
0

Unknown değerlerin sütunlara göre dağılımı:
job           330
marital        80
education    1731
default      8597
housing       990
loan          990
dtype: int64


In [30]:
print("Sayısal sütunların istatistikleri:")
display(df.describe())

print("\nKategorik sütunların istatistikleri:")
display(df.describe(include='object'))

Sayısal sütunların istatistikleri:


,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000



Kategorik sütunların istatistikleri:


C:\Users\nusayba\AppData\Local\Temp\ipykernel_38560\3892473361.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include='object'))


,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
count,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188
unique,12,4,8,3,3,3,2,10,5,3,2
top,admin.,married,university.degree,no,yes,no,cellular,may,thu,nonexistent,no
freq,10422,24928,12168,32588,21576,33950,26144,13769,8623,35563,36548


In [31]:
print("Hedef değişkenin sınıf dağılımı:")
print(df['y'].value_counts())

print("\nHedef değişkenin yüzdelik dağılımı:")
print(df['y'].value_counts(normalize=True) * 100)

Hedef değişkenin sınıf dağılımı:
y
no     36548
yes     4640
Name: count, dtype: int64

Hedef değişkenin yüzdelik dağılımı:
y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64


In [38]:
# pdays özel kod kontrolü
print("pdays = 999 olan kayıt sayısı:")
print((df['pdays'] == 999).sum())

# Yaş kontrolü
print("\nYaş aralığı:")
print("Minimum:", df['age'].min())
print("Maksimum:", df['age'].max())

# Campaign kontrolü
print("\nCampaign aralığı:")
print("Minimum:", df['campaign'].min())
print("Maksimum:", df['campaign'].max())

# Sayısal sütunlarda aykırı değerleri IQR yöntemiyle inceleme
numeric_columns = df.select_dtypes(include=['number']).columns

for column in numeric_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    alt_sinir = Q1 - 1.5 * IQR
    ust_sinir = Q3 + 1.5 * IQR

    aykiri_sayisi = (
        (df[column] < alt_sinir) |
        (df[column] > ust_sinir)
    ).sum()

    print(column, "aykırı değer sayısı:", aykiri_sayisi)

pdays = 999 olan kayıt sayısı:
39673

Yaş aralığı:
Minimum: 17
Maksimum: 98

Campaign aralığı:
Minimum: 1
Maksimum: 56
age aykırı değer sayısı: 469
duration aykırı değer sayısı: 2963
campaign aykırı değer sayısı: 2406
pdays aykırı değer sayısı: 1515
previous aykırı değer sayısı: 5625
emp.var.rate aykırı değer sayısı: 0
cons.price.idx aykırı değer sayısı: 0
cons.conf.idx aykırı değer sayısı: 447
euribor3m aykırı değer sayısı: 0
nr.employed aykırı değer sayısı: 0


In [39]:
# Mode ile doldurulacak kategorik sütunlar
mode_features = [
    'job',
    'marital',
    'education',
    'housing',
    'loan'
]

# Unknown değerlerini ayrı kategori olarak koruyacağımız sütun
unknown_features = ['default']

print("Mode ile doldurulacak sütunlar:", mode_features)
print("Unknown olarak korunacak sütunlar:", unknown_features)

Mode ile doldurulacak sütunlar: ['job', 'marital', 'education', 'housing', 'loan']
Unknown olarak korunacak sütunlar: ['default']


**Yorum:** (bu çıktı bize ne söylüyor?)

## D2. Kategorik değişkenlerin kodlanması

In [37]:
# Hedef değişken
y = df['y'].map({'yes': 1, 'no': 0})

# Modelde kullanılacak özellikler
# duration veri sızıntısı nedeniyle çıkarılıyor.
X = df.drop(columns=['y', 'duration'])

# Kategorik sütunlar
categorical_features = X.select_dtypes(
    include=['object', 'str']
).columns.tolist()

# Sayısal sütunlar
numeric_features = X.select_dtypes(
    include=['number']
).columns.tolist()

print("Kategorik sütunlar:")
print(categorical_features)

print("\nSayısal sütunlar:")
print(numeric_features)

print("\nKategorik sütun sayısı:", len(categorical_features))
print("Sayısal sütun sayısı:", len(numeric_features))

print("\nX boyutu:", X.shape)
print("y boyutu:", y.shape)

Kategorik sütunlar:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Sayısal sütunlar:
['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

Kategorik sütun sayısı: 10
Sayısal sütun sayısı: 9

X boyutu: (41188, 19)
y boyutu: (41188,)


**Yorum:** (bu çıktı bize ne söylüyor?)

## D3. Normalizasyon / standardizasyon

In [33]:
scaler = StandardScaler()

print("StandardScaler oluşturuldu.")

StandardScaler oluşturuldu.


**Yorum:** (bu çıktı bize ne söylüyor?)

## D6. Pipeline ve ColumnTransformer

In [42]:
# Sayısal veriler için ön işleme
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Mode ile doldurulacak sütunların işlemleri
mode_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Unknown kategorisini koruyacak sütunun işlemleri
unknown_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('mode', mode_transformer, mode_features),
        ('unknown', unknown_transformer, unknown_features),
        ('other_cat', OneHotEncoder(handle_unknown='ignore'), [
            col for col in categorical_features
            if col not in mode_features + unknown_features
        ])
    ]
)

print("ColumnTransformer başarıyla oluşturuldu!")

ColumnTransformer başarıyla oluşturuldu!


**Yorum:** (bu çıktı bize ne söylüyor?)

## D7. Eğitim/test bölmesi

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Eğitim verisi boyutu:")
print(X_train.shape)

print("\nTest verisi boyutu:")
print(X_test.shape)

print("\nEğitim hedef dağılımı:")
print(y_train.value_counts())

print("\nTest hedef dağılımı:")
print(y_test.value_counts())

Eğitim verisi boyutu:
(32950, 19)

Test verisi boyutu:
(8238, 19)

Eğitim hedef dağılımı:
y
0    29238
1     3712
Name: count, dtype: int64

Test hedef dağılımı:
y
0    7310
1     928
Name: count, dtype: int64


**Yorum:** (bu çıktı bize ne söylüyor?)

Ön İşleme Son Kontrolü

In [36]:
print("========== ÖN İŞLEME SON KONTROLÜ ==========")

print("\nVeri seti:", df.shape)
print("Eğitim verisi:", X_train.shape)
print("Test verisi:", X_test.shape)

print("\nKategorik özellik sayısı:", len(categorical_features))
print("Sayısal özellik sayısı:", len(numeric_features))

print("\nEğitim sınıf oranları:")
print(y_train.value_counts(normalize=True))

print("\nTest sınıf oranları:")
print(y_test.value_counts(normalize=True))

print("\nÖn işleme tamamlandı!")

========== ÖN İŞLEME SON KONTROLÜ ==========

Veri seti: (41188, 21)
Eğitim verisi: (32950, 19)
Test verisi: (8238, 19)

Kategorik özellik sayısı: 10
Sayısal özellik sayısı: 9

Eğitim sınıf oranları:
y
0    0.887344
1    0.112656
Name: proportion, dtype: float64

Test sınıf oranları:
y
0    0.887351
1    0.112649
Name: proportion, dtype: float64

Ön işleme tamamlandı!
